## Simple imputer: Small explanation is in the file 'all_about_it.md' and mostly explained here with code executions
Simple imputer would be recommended when features are mostly independent (no corelation between each other mostly). 
- WHen it comes to computation speed, this is very fast O(N)
- Works well with smaller and larger datasets.

But also comes with some cons
- when its comes to accuracy, its less for complex datasets
- Sensitive for outliers (if using `mean`).

But you can go with this for learning purpose, as we use smaller datasets and its fast to use.

### When to use `mean`, when to use `median` and when to use `mode`.
This question has a lot of explanation beyond it, it starts with knowing how is our data distributed. We need to know whether our data is normally distributed or is it skewed. And let me first explain you what is that skewed or normally distributed thing.

The very first and simple way would by using box plot for your data. which will tell you explain you everything. And also i'll include the other way I know below this/

Below is the code which generates a diagram with some data and later will be the explanation, including the things mentioned above.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

# Set seed for reproducibility
np.random.seed(42)

# 1. Symmetric (Normal) Distribution (e.g., Heights, Test Scores)
symmetric_data = np.random.normal(loc=50, scale=10, size=1000)

# 2. Right-Skewed (Positive Skew) Distribution (e.g., Income, House Prices)
right_skewed_data = np.random.exponential(scale=15, size=1000)

# 3. Left-Skewed (Negative Skew) Distribution (e.g., Age at Death, Retirement Age)
left_skewed_data = 100 - np.random.exponential(scale=15, size=1000)

# Combine into a pandas DataFrame
df = pd.DataFrame({
    'Symmetric (Normal)': symmetric_data,
    'Right-Skewed': right_skewed_data,
    'Left-Skewed': left_skewed_data
})

# Plotting Box Plots and Histograms/KDEs
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

cols = ['Symmetric (Normal)', 'Right-Skewed', 'Left-Skewed']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, col in enumerate(cols):
    # Row 1: Box Plot
    sns.boxplot(x=df[col], ax=axes[0, i], color=colors[i])
    axes[0, i].set_title(f'Box Plot: {col}', fontsize=12, fontweight='bold')
    axes[0, i].set_xlabel('')
    
    # Row 2: Histogram + KDE
    sns.histplot(df[col], kde=True, ax=axes[1, i], color=colors[i], bins=30)
    axes[1, i].set_title(f'Histogram & KDE: {col}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Display summary statistics
stats_summary = pd.DataFrame({
    'Mean': df.mean(),
    'Median': df.median(),
    'Skewness': df.skew()
})
print(stats_summary)

The above code will generate a image which is located in the path: [data_preprocessing\1) missing_data_handling\1) simple_imputer\Code_Generated_Image.png]

#### Its statistical summary comparison:
| **Dataset Case**       | **Mean** | **Median** | **Skewness Score** | **Relationship**                    | **Recommended Imputer**            |
| ---------------------- | -------- | ---------- | ------------------ | ----------------------------------- | ---------------------------------- |
| **Symmetric (Normally distributed)** | 50.19    | 50.25      | **0.12**           | $\text{Mean} \approx \text{Median}$ | `SimpleImputer(strategy='mean')`   |
| **Right-Skewed**       | 15.12    | 10.89      | **1.98**           | $\text{Mean} > \text{Median}$       | `SimpleImputer(strategy='median')` |
| **Left-Skewed**        | 85.40    | 89.85      | **-1.64**          | $\text{Mean} < \text{Median}$       | `SimpleImputer(strategy='median')` |

How to Identify Each Case Visually:

##### 1. Symmetric (Normal)
- Box Plot: The median line is centered inside the box. Both whiskers extending to the left and right are equal in length, and any minor outliers are spread symmetrically on both sides.
- Histogram/KDE: Forms a balanced, classic bell-curve shape centered around 50.
- Imputation Choice: strategy='mean'. Since the mean and median are virtually identical, the mean incorporates all available data without being pulled by extreme values.

##### 2. Right-Skewed (Positively Skewed)
- Box Plot: The median line sits towards the left side of the box. The right whisker is significantly longer, and outliers extend far to the high-value (right) end.
- Histogram/KDE: The peak is concentrated on the left side (lower numbers), with a long "tail" stretching towards the right (e.g., household income).
- Imputation Choice: strategy='median'. High outliers pull the mean upwards (15.12 vs median of 10.89). The median preserves the true central tendency.

##### 3. Left-Skewed (Negatively Skewed)
- Box Plot: The median line sits towards the right side of the box. The left whisker is much longer, and outliers stretch out towards the lower values on the left.
- Histogram/KDE: The peak is concentrated on the right side (higher numbers), with a long tail stretching towards the left (e.g., age at retirement).
- Imputation Choice: strategy='median'. Low outliers pull the mean downwards (85.40 vs median of 89.85).

---
I hope you understood this way, as its basic statistics.

#### Now the other way I prefer is using: **Pandas method that computes the Adjusted Fisher-Pearson Coefficient of Skewness. The .skew() function**

Instead of understanding some complex formulas, pandas can calculate that. Here's how to do that and what do you get.

First you find skewness of the columns you want, here is a code snipper for that:
```python
import pandas as pd

# 1. Get the skewness score of a single column
income_skew = df['income'].skew()
print(f"Income Skewness: {income_skew}")

# 2. Get skewness scores for ALL numerical columns in a DataFrame
all_skews = df.skew(numeric_only=True)
print(all_skews)
```

***How to Interpret the Output Value***

When you print the output of .skew(), Pandas returns a single floating-point number:

| **Output Value**                         | **Interpretation**  | **Distribution Shape**          | **Recommended SimpleImputer** |
| ---------------------------------------- | ------------------- | ------------------------------- | ----------------------------- |
| **Close to $0$** (e.g., $-0.2$ to $0.2$) | Virtually symmetric | Normal / Bell-shaped            | `strategy='mean'`             |
| **Greater than $+0.5$`**                 | Positively skewed   | Tail stretches to the **right** | `strategy='median'`           |
| **Less than $-0.5$`**                    | Negatively skewed   | Tail stretches to the **left**  | `strategy='median'`           |


And mode is used for categorical data.



### How does it work
this process includes two things:
1) .fit(): which finds all the missing values locations from data
2) .transform(): applies the transformation. as we decided to fill the cells with mean, it fills that cells with mean of that dataset.
3) .fit_transform(): simply the combination of both of these.

as this is just example data, but even if you take larger dataset, the process remains same, but you just need to change which row or column or cell should be transformed

In [ ]:
# example data
import numpy as np
data = np.array([[2.3, np.nan], [np.nan, 3], [4, 5]])
print("Original data:\n", data)

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
imputer.fit(data)
print("transformed_data:\n",imputer.transform(data))